### Packages

In [10]:
import pickle
import numpy as np
import math
import matplotlib.pyplot as plt
import torch
import tarfile
import urllib.request
import os
import shutil


In [16]:

FILEPATH="Datasets/cifar-10-batches-py/" # we add the directory path first 


### Read the file, this will be used to access the train, val and test batches.

In [28]:

def LoadBatch(filename):
    with open(filename, "rb") as fo:
        batch = pickle.load(fo, encoding="bytes")

    X = batch[b"data"].astype(np.float64).T / 255.0   # 3072 x 10000  divide by 255.0 so to keep everything small 
    y = np.array(batch[b"labels"])                    # length 10000, since we have 1000 images

    K = 10
    n = X.shape[1]
    Y = np.zeros((K, n), dtype=X.dtype)
    Y[y, np.arange(n)] = 1 # each column represent the class representation of one image 

    return X, Y, y

 ### Normalize the data: X_train, X_validation, X_test

In [46]:
def normalize(X, mean_X, std_X):
    return (X - mean_X) / std_X

### Access the training, validation and test data.

In [47]:


X_train,Y_train,y_train=LoadBatch(FILEPATH+"data_batch_1")

X_validation,Y_validation,y_validation=LoadBatch(FILEPATH+"data_batch_2")

X_test,Y_test,y_test=LoadBatch(FILEPATH+"test_batch")

d = X_train.shape[0]

mean_X = np.mean(X_train, axis=1).reshape(d, 1)
std_X  = np.std(X_train, axis=1).reshape(d, 1)

X_train = normalize(X_train, mean_X, std_X)
X_validation = normalize(X_validation, mean_X, std_X)
X_test = normalize(X_test, mean_X, std_X)

### Initialize parameters $W$ and $b$

In [57]:
def init_network(X_train):

    K = 10
    d = X_train.shape[0]

    rng = np.random.default_rng()
    BitGen = type(rng.bit_generator)
    seed = 42
    rng.bit_generator.state = BitGen(seed).state

    init_net = {
        'W': 0.01 * rng.standard_normal((K, d)),
        'b': np.zeros((K, 1))
    }
    return init_net



### Softmax function

In [52]:
def softmax(s):
    return np.exp(s)/np.sum(np.exp(s),axis=0,keepdims=True)

### Define a function that applies:

\begin{equation*}
\begin{aligned}
\mathbf{s} & =W \mathbf{x}+\mathbf{b} \\
\mathbf{p} & =\operatorname{SOFTMAX}(\mathbf{s})
\end{aligned}
\end{equation*}

In [60]:
def ApplyNetwork(X,network):


    W=network['W']
    b=network['b']

    s=np.dot(W,X)
    P=softmax(s)
    return P

In [63]:
network=init_network(X_train)
P = ApplyNetwork(X_train[:, 0:100], network)
print(P.shape)
print(np.sum(P[:, 0]))

(10, 100)
1.0


### Compute

\begin{equation*}
J(\mathcal{D}, \lambda, W, b)=\frac{1}{|\mathcal{D}|} \sum_{(\mathbf{x}, y) \in \mathcal{D}} -\log \left(p_y\right)
\end{equation*}



In [ ]:
def ComputeLoss(P, y):
    n = P.shape[1]
    correct_class_probs = P[y, np.arange(n)]
    L = -np.mean(np.log(correct_class_probs))
    return L

